In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "DOTUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,vol_regime_ratio,hour_sin,hour_cos,dow_sin,dow_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,4.077,4.077,4.064,4.066,5847.91,2025-06-01 00:04:59.999999+00:00,23798.52466,176,2582.80,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,4.066,4.067,4.063,4.066,4229.80,2025-06-01 00:09:59.999999+00:00,17192.67006,126,2434.04,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000,0.000000,NaN,NaN
2,2025-06-01 00:10:00+00:00,4.065,4.066,4.054,4.057,18409.90,2025-06-01 00:14:59.999999+00:00,74705.84886,278,4347.71,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000718,-0.000144,-0.000574,NaN,NaN
3,2025-06-01 00:15:00+00:00,4.058,4.058,4.049,4.053,9032.44,2025-06-01 00:19:59.999999+00:00,36596.15019,219,4481.75,...,NaN,0.0,1.0,-0.781831,0.62349,-0.001591,-0.000433,-0.001158,NaN,NaN
4,2025-06-01 00:20:00+00:00,4.053,4.059,4.051,4.057,7298.53,2025-06-01 00:24:59.999999+00:00,29591.97128,152,4427.02,...,NaN,0.0,1.0,-0.781831,0.62349,-0.001938,-0.000734,-0.001204,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,438
[info] optuna train rows: 53,400
[info] valid rows:        13,350
[info] test rows:         16,688


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 15:26:29,523] A new study created in memory with name: no-name-57cb8e02-9167-43d6-b100-35e2bc80b2f7


[I 2026-03-23 15:26:29,717] Trial 0 finished with value: 0.5320337390512804 and parameters: {'n_estimators': 500, 'learning_rate': 0.046187109390049115, 'max_depth': 5, 'subsample': 0.7996646210492592, 'colsample_bytree': 0.6890046601106091, 'colsample_bylevel': 0.6889986300840507, 'min_child_weight': 5, 'gamma': 2.5985284373248057, 'reg_alpha': 0.12306931514988033, 'reg_lambda': 8.341106432362084, 'scale_pos_weight': 0.9719677016911182}. Best is trial 0 with value: 0.5320337390512804.


[I 2026-03-23 15:26:29,962] Trial 1 finished with value: 0.5396838774891506 and parameters: {'n_estimators': 900, 'learning_rate': 0.03818145165896871, 'max_depth': 3, 'subsample': 0.6954562418017751, 'colsample_bytree': 0.6958511274633585, 'colsample_bylevel': 0.7260605607398845, 'min_child_weight': 13, 'gamma': 1.2958350559263474, 'reg_alpha': 0.010295300642650052, 'reg_lambda': 6.252287916406214, 'scale_pos_weight': 1.0068961895724917}. Best is trial 1 with value: 0.5396838774891506.


[I 2026-03-23 15:26:30,267] Trial 2 finished with value: 0.535605726169043 and parameters: {'n_estimators': 500, 'learning_rate': 0.018033330377234345, 'max_depth': 4, 'subsample': 0.8462939903482534, 'colsample_bytree': 0.6999184455395899, 'colsample_bylevel': 0.778558609603403, 'min_child_weight': 14, 'gamma': 0.13935123815999317, 'reg_alpha': 0.12957079329680446, 'reg_lambda': 1.6666983286066417, 'scale_pos_weight': 0.984885354394028}. Best is trial 1 with value: 0.5396838774891506.


[I 2026-03-23 15:26:30,455] Trial 3 finished with value: 0.5320034814396655 and parameters: {'n_estimators': 900, 'learning_rate': 0.047309442068985116, 'max_depth': 5, 'subsample': 0.7261534422933427, 'colsample_bytree': 0.6744180285015959, 'colsample_bylevel': 0.8210582566280392, 'min_child_weight': 12, 'gamma': 0.3661147045343365, 'reg_alpha': 0.05269751777340593, 'reg_lambda': 1.1085122517311703, 'scale_pos_weight': 1.2654663424992434}. Best is trial 1 with value: 0.5396838774891506.


...

[I 2026-03-23 15:27:00,885] Trial 96 finished with value: 0.5340833119050844 and parameters: {'n_estimators': 800, 'learning_rate': 0.027465013289107314, 'max_depth': 4, 'subsample': 0.7230369503785199, 'colsample_bytree': 0.7721421934501697, 'colsample_bylevel': 0.7952709176785542, 'min_child_weight': 8, 'gamma': 1.4323876208951654, 'reg_alpha': 0.011389406992804527, 'reg_lambda': 6.403140773408892, 'scale_pos_weight': 1.10171724197074}. Best is trial 63 with value: 0.5402799806107873.


[I 2026-03-23 15:27:01,337] Trial 97 finished with value: 0.5368740891867275 and parameters: {'n_estimators': 600, 'learning_rate': 0.010954532587998384, 'max_depth': 4, 'subsample': 0.6684929185834129, 'colsample_bytree': 0.765305460283433, 'colsample_bylevel': 0.7658178856452135, 'min_child_weight': 15, 'gamma': 2.7165487048310695, 'reg_alpha': 0.006907304615464338, 'reg_lambda': 4.851194259469632, 'scale_pos_weight': 1.0882396814110478}. Best is trial 63 with value: 0.5402799806107873.


[I 2026-03-23 15:27:01,578] Trial 98 finished with value: 0.5377198598634213 and parameters: {'n_estimators': 900, 'learning_rate': 0.04063234138954459, 'max_depth': 4, 'subsample': 0.6547471410186052, 'colsample_bytree': 0.6930520743628061, 'colsample_bylevel': 0.8358040792627394, 'min_child_weight': 5, 'gamma': 2.843799042967045, 'reg_alpha': 0.0013644792097697778, 'reg_lambda': 4.168698461722295, 'scale_pos_weight': 1.164514118583242}. Best is trial 63 with value: 0.5402799806107873.


[I 2026-03-23 15:27:01,866] Trial 99 finished with value: 0.5365325331530453 and parameters: {'n_estimators': 800, 'learning_rate': 0.02264439086896065, 'max_depth': 4, 'subsample': 0.6795432504848155, 'colsample_bytree': 0.751237452375897, 'colsample_bylevel': 0.8028046575315787, 'min_child_weight': 13, 'gamma': 2.135921306929989, 'reg_alpha': 0.0011365386580756954, 'reg_lambda': 3.741537788771621, 'scale_pos_weight': 0.9971981293122957}. Best is trial 63 with value: 0.5402799806107873.


['dist_ma_30', 'dow_cos', 'vol_30', 'dow_sin', 'hour_cos', 'hour_sin', 'mom_60', 'atr_norm', 'vol_regime_ratio', 'trend_strength', 'macd_hist', 'mom_15', 'imbalance_15', 'range_ratio', 'vol_ratio_5_30', 'dist_ma_15', 'vol_5', 'mom_5', 'bar_range', 'trades_z', 'co_spread', 'num_trades_mom_5', 'volume_z', 'imbalance_z', 'volume_mom_5']
feature
dist_ma_30          9.931702
dow_cos             9.841445
vol_30              9.570422
dow_sin             9.563879
hour_cos            9.160274
hour_sin            9.071434
mom_60              9.047112
atr_norm            8.671803
vol_regime_ratio    8.485468
trend_strength      8.455782
macd_hist           8.438039
mom_15              8.434170
imbalance_15        8.357536
range_ratio         8.017530
vol_ratio_5_30      8.009110
dist_ma_15          7.760035
vol_5               7.725232
mom_5               7.515078
bar_range           7.068305
trades_z            6.920411
co_spread           6.807412
num_trades_mom_5    6.768508
volume_z          

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.156050
Test IC:         0.054577
Train ROC AUC:   0.591232
Test ROC AUC:    0.532869
Train PR AUC:    0.565247
Test PR AUC:     0.486464
Train Log Loss:  0.686699
Test Log Loss:   0.693337
Train Brier:     0.246792
Test Brier:      0.250092
Train Accuracy:  0.553603
Test Accuracy:   0.507371
Train Precision: 0.527350
Test Precision:  0.469505
Train Recall:    0.663135
Test Recall:     0.659799
Train F1:        0.587499
Test F1:         0.548619


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.429, 0.478] -0.000341   1669  0.006978
(0.478, 0.488] -0.000430   1669  0.006589
(0.488, 0.495] -0.000135   1669  0.006294
(0.495, 0.503] -0.000162   1668  0.005704
(0.503, 0.508] -0.000143   1669  0.005696
(0.508, 0.514]  0.000002   1669  0.006242
(0.514, 0.519] -0.000247   1668  0.005822
(0.519, 0.527] -0.000049   1669  0.005772
(0.527, 0.541]  0.000079   1669  0.007336
(0.541, 0.69]   0.000839   1669  0.012974


/tmp/ipykernel_1542188/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/DOTUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/DOTUSDT__h6_model.joblib
[saved] features -> models/xgb/DOTUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/DOTUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/DOTUSDT__h6_meta.json
